# Étape 3 — Chaîne RAG complète (retriever + LLM Ollama/Llama 3)

**Objectif de ce notebook :**
- Recharger l'index FAISS construit à l'étape 2
- Vérifier qu'Ollama tourne et qu'un modèle est disponible
- Construire le prompt final (contexte + question + consignes)
- Appeler le LLM et obtenir une vraie réponse en langage naturel
- Comparer avec ce que renvoyait la simple recherche de similarité (étape 2)

**Prérequis :** Ollama installé et un modèle téléchargé (`ollama pull llama3` ou
`ollama pull llama3.2:3b`).

**Kernel à utiliser :** `Python (RAG GED)`.

## 0. Imports et accès au dossier `src/`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

## 3.0 Vérifier qu'Ollama tourne et quels modèles sont disponibles

Avant d'aller plus loin, on vérifie que le service Ollama répond bien et qu'au moins
un modèle est téléchargé. Si cette cellule affiche une erreur, retourne dans un
terminal PowerShell et lance :
```
ollama pull llama3.2:3b
```
(ou `ollama pull llama3` si tu préfères le modèle plus complet — plus lent à télécharger
et à faire tourner).

In [2]:
import ollama

try:
    models_response = ollama.list()
    model_names = [m.model for m in models_response.models]
    if not model_names:
        print("⚠️ Aucun modèle Ollama n'est encore téléchargé.")
        print("Lance dans un terminal : ollama pull llama3.2:3b")
    else:
        print("Modèles Ollama disponibles :", model_names)
except Exception as exc:
    print("⚠️ Impossible de contacter Ollama. Vérifie qu'il est bien lancé.")
    print("Erreur :", exc)

Modèles Ollama disponibles : ['llama3.2:3b', 'aya:8b', 'gemma3:latest']


## 3.1 Choisir le modèle à utiliser

Adapte `OLLAMA_MODEL` ci-dessous en fonction de ce que la cellule précédente a affiché
(ex : `"llama3"`, `"llama3.2:3b"`, `"llama3:latest"`...).

In [3]:
OLLAMA_MODEL = "llama3.2:3b"  # <-- adapte si besoin selon la liste ci-dessus

## 3.2 Recharger l'index FAISS (construit au notebook 2)

In [4]:
from build_index import EMBEDDING_MODEL_NAME, INDEX_DIR
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
vectorstore = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Index FAISS rechargé depuis :", INDEX_DIR)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Index FAISS rechargé depuis : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\faiss_index


## 3.3 Construire le prompt

Le prompt indique explicitement au LLM de répondre UNIQUEMENT à partir du contexte
fourni, et de dire clairement quand il ne trouve pas la réponse plutôt que d'inventer.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """Tu es un assistant interne pour les opérateurs de GDS GED Services,
une entreprise de Gestion Électronique des Documents (GED). Réponds à la question de
l'opérateur UNIQUEMENT à partir du contexte fourni ci-dessous, extrait de la
documentation interne (procédures, ateliers, logiciels utilisés).

Règles :
- Si la réponse ne se trouve pas dans le contexte, dis clairement que tu ne trouves pas
  l'information dans la documentation disponible, ne l'invente pas.
- Réponds de façon claire, concise et professionnelle, en français.
- Si utile, structure ta réponse en étapes ou en liste.

Contexte :
{context}

Question : {question}

Réponse :"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
print("Prompt prêt.")

Prompt prêt.


## 3.4 Construire la chaîne complète (retriever → prompt → LLM)

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.2)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Chaîne RAG construite avec le modèle :", OLLAMA_MODEL)

Chaîne RAG construite avec le modèle : llama3.2:3b


## 3.5 Tester avec une vraie question

⏱️ La première génération peut prendre un peu de temps (le modèle se charge en mémoire).
Les suivantes seront plus rapides.

In [7]:
question = "Comment fonctionne l'atelier Préparation ?"

docs_utilises = retriever.invoke(question)
reponse = chain.invoke(question)

print("Question :", question)
print("\nRéponse générée :\n")
print(reponse)

print("\n--- Sources utilisées ---")
for doc in docs_utilises:
    print("-", doc.metadata.get("source"))

Question : Comment fonctionne l'atelier Préparation ?

Réponse générée :

L'atelier Préparation consiste en plusieurs étapes :

1. Ouvrir chaque dossier du lot et classer les pièces à l'intérieur du dossier dans le bon ordre.
2. Vérifier la complétude du dossier (toutes les pièces attendues sont présentes).
3. Retirer les agrafes et trombones qui pourraient endommager le scanner.

Cet atelier est crucial pour garantir que les dossiers soient prêts à être numérisés de manière efficace et avec une qualité élevée. Il permet également de détecter les erreurs potentielles, telles que le classement accidentel d'une page dans le mauvais dossier, qui peuvent être détectées automatiquement ou manuellement lors du contrôle qualité.

--- Sources utilisées ---
- C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
- C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\exemple_atelier_preparation.md
- C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\d

In [8]:
# Essaie avec d'autres questions
question_2 = "Quelle est la différence entre PEC et RD ?"
reponse_2 = chain.invoke(question_2)

print("Question :", question_2)
print("\nRéponse générée :\n")
print(reponse_2)

Question : Quelle est la différence entre PEC et RD ?

Réponse générée :

La réponse à votre question est :

PEC (Prise En Charge) et RD (Refus de Dossier) sont deux types de décisions possibles à l'issue du traitement d'un dossier. La principale différence entre ces deux termes réside dans la nature de la décision prise :

- PEC (Prise En Charge) : le dossier est accepté et pris en charge par l'organisme concerné.
- RD (Refus de Dossier) : le dossier est refusé, généralement parce qu'il est incomplet ou ne respecte pas les critères requis.

En d'autres termes, si un dossier est classifié sous PEC, cela signifie que l'organisme a accepté de traiter et de prendre en charge ce dossier. En revanche, si un dossier est classifié sous RD, cela signifie qu'il n'a pas été accepté pour traitement ou qu'il ne répond pas aux exigences requises.


In [9]:
# Teste aussi une question dont la réponse N'EST PAS dans tes documents,
# pour vérifier que le LLM dit bien qu'il ne sait pas plutôt que d'inventer
question_3 = "Quel est le salaire moyen des opérateurs GED ?"
reponse_3 = chain.invoke(question_3)

print("Question :", question_3)
print("\nRéponse générée :\n")
print(reponse_3)

Question : Quel est le salaire moyen des opérateurs GED ?

Réponse générée :

Je ne trouve pas d'information dans la documentation disponible concernant le salaire moyen des opérateurs GED. La documentation fournie ne contient aucune information sur les salaires ou les tarifs payés aux employés de GDS GED Services, y compris les opérateurs GED.


**Ce qu'il faut vérifier ici :**
- Les réponses aux questions 1 et 2 doivent être cohérentes avec le contenu de tes
  documents (compare avec les chunks vus au notebook 2).
- La réponse à la question 3 doit clairement indiquer que l'information n'est pas
  disponible, **pas** une réponse inventée. Si le LLM invente quand même une réponse,
  c'est un signe qu'il faut renforcer le prompt (section 3.3) ou changer de modèle.

## Conclusion et prochaine étape

Tu as maintenant un pipeline RAG complet et fonctionnel de bout en bout :
`chunking → embeddings → FAISS → retriever → prompt → LLM → réponse`.

Le code de ce notebook correspond exactement à la classe `RagAssistant` dans
`src/rag_chain.py` — rien à recopier, c'est déjà prêt.

**Prochaine étape :** brancher tout ça derrière une API et une interface web :

```powershell
# Terminal 1 — l'API
uvicorn src.api:app --reload --port 8000

# Terminal 2 — l'interface
streamlit run app_streamlit.py
```

Si `OLLAMA_MODEL` n'est pas `"llama3"` (ex : tu utilises `"llama3.2:3b"`), pense à
mettre à jour cette même valeur dans `src/rag_chain.py` avant de lancer l'API.